# Les règles du jeu : User-Agent et robots.txt
**ÉNONCÉ**

Dans le notebook précédent, votre script a parcouru `books.toscrape.com` : un site qui **invite** à être scrapé. Sur la plupart des sites, **deux notions** décident si votre programme est le bienvenu ou bloqué :

| Notion | En une phrase |
|---|---|
| **User-Agent** | L'étiquette que votre programme envoie pour dire **qui il est** |
| **robots.txt** | Le panneau d'accès du site : **où** les robots ont le droit d'aller |

Ici, vous les **découvrez en les testant**, sur de vrais sites, puis vous les ajoutez à votre script.

**Ce qui est fait :** étapes (à lire et exécuter). **Ce que vous complétez :** étapes (un `___` ou un `# TODO`), suivies d'un Contrôle.

In [45]:
import time
import requests
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser
from bs4 import BeautifulSoup

LIVRES = "https://books.toscrape.com"      # le site d'entraînement du notebook précédent
WIKI = "https://en.wikipedia.org"          # un site qui n'est PAS fait pour être scrapé à la légère

def controle(condition, ok, ko):
    print("✅", ok) if condition else print("❌", ko)
    
    return condition
controle
print(controle(True, "Tout va bien", "Il y a un problème"))

✅ Tout va bien
True


## Étape 1 — Ce que votre script envoie, sans que vous le voyiez
Chaque fois que `requests.get` télécharge une page, il envoie au site, **en plus de l'adresse**, une série d'informations : les **en-têtes**. L'un d'eux s'appelle le **User-Agent** : c'est l'étiquette qui dit **qui** fait la demande (un navigateur, un robot, un programme…).

Regardons ce que **votre** script envoie, sans rien avoir précisé.

In [30]:
reponse = requests.get(LIVRES + "/")
print("Votre User-Agent actuel :", reponse.request.headers["User-Agent"])
print("\nTous les en-têtes envoyés :")
for nom, valeur in reponse.request.headers.items():
    print("  ", nom, ":", valeur)

Votre User-Agent actuel : python-requests/2.34.2

Tous les en-têtes envoyés :
   User-Agent : python-requests/2.34.2
   Accept-Encoding : gzip, deflate
   Accept : */*
   Connection : keep-alive


## Étape 2 — Le même script sur un site qui n'est pas fait pour ça
Votre script marchait sur `books.toscrape.com` parce que ce site **invite** à être scrapé. Essayons le même geste, **sans rien changer**, sur Wikipédia.

**Notez ce que vous obtenez** : le résultat peut varier d'un site à l'autre et d'un moment à l'autre, et c'est précisément ce qu'on veut observer.

| Code | Sens |
|---|---|
| **200** | Accepté |
| **403** | Refusé : le serveur a compris votre demande mais ne veut pas y répondre |
| **429** | Trop de demandes : vous allez trop vite |

In [31]:
sans_presentation = requests.get(WIKI + "/wiki/Astronomy")
print("Sans vous présenter : code", sans_presentation.status_code)

Sans vous présenter : code 403


## Étape 3 — Se présenter : votre User-Agent
Wikimedia demande aux programmes de **se présenter avec un moyen de les contacter**. Son format recommandé : `nom du projet/version (contact) bibliothèque`. Le contact peut être **une adresse e-mail ou l'adresse d'une page de projet**. Cette adresse est visible par les administrateurs du site : n'utilisez pas une adresse privée.

In [32]:
ENTETES = {"User-Agent": "TPFormation/1.0 (___) python-requests"}   # TODO : remplacez ___ par votre contact
print(ENTETES)

{'User-Agent': 'TPFormation/1.0 (___) python-requests'}


In [33]:
#  Contrôle (ne pas modifier)
ua = ENTETES['User-Agent']
controle(('@' in ua or 'http' in ua) and '___' not in ua and not any(m in ua.lower() for m in ('example', 'exemple', 'votre.')), 'User-Agent renseigné : ' + ua, 'Remplacez ___ par un vrai contact (e-mail ou adresse de page), et non par une adresse d\'exemple')

❌ Remplacez ___ par un vrai contact (e-mail ou adresse de page), et non par une adresse d'exemple


## Étape 4 — Le même site, en vous présentant
Refaites exactement la même demande, mais cette fois avec votre carte de visite.

In [34]:
avec_presentation = requests.get(WIKI + "/wiki/Astronomy", headers=ENTETES)
print("Sans vous présenter  : code", sans_presentation.status_code)
print("En vous présentant   : code", avec_presentation.status_code)

Sans vous présenter  : code 403
En vous présentant   : code 200


### Question
**Que s'est-il passé entre les deux demandes ? Si les deux codes sont identiques, qu'en concluez-vous, et pourquoi faut-il tout de même vous présenter ?**

*Votre réponse (2 à 4 lignes) :*

> 

## Étape 5 — Le panneau d'accès : robots.txt
Chaque site peut publier, à une adresse fixe (`nom-du-site/robots.txt`), un petit fichier texte qui dit **quelles zones les robots peuvent visiter**. Exemple de lecture :

```
User-agent: *
Disallow: /prive/
```
« Pour tous les robots (`*`), ne pas aller dans `/prive/`. »

Ce n'est **pas une barrière technique** : votre programme *pourrait* y aller. C'est une règle que les collecteurs sérieux respectent. La fonction ci-dessous télécharge le fichier et le transforme en un objet qui sait répondre à la question « ai-je le droit de visiter cette adresse ? ».

In [35]:
def lire_robots(base):
    reponse = requests.get(base + "/robots.txt", headers=ENTETES)
    print(base, "-> code", reponse.status_code)
    if reponse.status_code == 404:
        print("   pas de fichier robots.txt : aucune restriction annoncée")
        return None
    if reponse.status_code != 200:
        raise RuntimeError("Réponse inattendue pour robots.txt : on s'abstient par prudence")
    regles = RobotFileParser()
    regles.parse(reponse.text.splitlines())
    print(reponse.text[:500])
    return regles

## Étape 6 — Demander la permission, adresse par adresse
Pour chaque adresse, on pose la question à l'objet de règles. Si le site n'a pas de `robots.txt`, aucune restriction n'est annoncée.

In [36]:
autorise = {}
for base, chemins in ((LIVRES, ["/catalogue/page-2.html", "/catalogue/category/books/travel_2/index.html"]),
                      (WIKI, ["/wiki/Astronomy", "/wiki/Special:Search", "/w/index.php?search=test"])):
    regles = lire_robots(base)
    for chemin in chemins:
        reponse_regle = True if regles is None else None      # TODO : poser la question avec regles.can_fetch(User-Agent, adresse complète)
        autorise[base + chemin] = reponse_regle
        print("  ", "OUI" if reponse_regle else "NON", chemin)

https://books.toscrape.com -> code 404
   pas de fichier robots.txt : aucune restriction annoncée
   OUI /catalogue/page-2.html
   OUI /catalogue/category/books/travel_2/index.html
https://en.wikipedia.org -> code 200
﻿# robots.txt for http://www.wikipedia.org/ and friends
#
# Please note: There are a lot of pages on this site, and there are
# some misbehaved spiders out there that go _way_ too fast. If you're
# irresponsible, your access to the site may be blocked.
#

# Observed spamming large amounts of https://en.wikipedia.org/?curid=NNNNNN
# and ignoring 429 ratelimit responses, claims to respect robots:
# http://mj12bot.com/
User-agent: MJ12bot
Disallow: /

# advertising-related bots:
User-agent: Mediapa
   NON /wiki/Astronomy
   NON /wiki/Special:Search
   NON /w/index.php?search=test


In [37]:
# ✅ Contrôle (ne pas modifier)
controle(len(autorise) == 5 and all(isinstance(v, bool) for v in autorise.values()), 'Les 5 adresses ont reçu une réponse OUI ou NON', 'Attendu : un booléen (True ou False) pour chacune des 5 adresses')

❌ Attendu : un booléen (True ou False) pour chacune des 5 adresses


### Question
**Lisez le robots.txt de Wikipédia affiché plus haut. Quelles zones sont interdites aux robots ? Votre script du notebook précédent aurait-il pu y aller, techniquement ? Aurait-il dû ?**

*Votre réponse (2 à 4 lignes) :*

> 

## Étape 7 — Faire une pause : une fonction de requête polie
Troisième réflexe : **ne pas envoyer 1 000 demandes à la suite**, comme une attaque. On ajoute une **pause** avant chaque requête, on **se présente**, et si le serveur répond « trop de demandes » (429) ou « indisponible » (503), on **attend de plus en plus longtemps** avant de réessayer : 2 secondes, puis 4, puis 8.

In [38]:
PAUSE = 1.0

def get_poli(url, essais=3):
    for n in range(essais):
        time.sleep(PAUSE)                                       # pause before each request
        reponse = requests.get(
            url,
            headers={'User-Agent': 'MonScraper/1.0 (contact: ton.email@exemple.com)'},
            timeout=20,
        )
        if reponse.status_code in (429, 503):                   # "too many requests" and "unavailable"
            time.sleep(2 ** (n + 1))
            continue
        return reponse
    raise RuntimeError("Échec après " + str(essais) + " essais : " + url)

In [39]:
# Contrôle (ne pas modifier)
debut_test = time.time(); essai = get_poli(LIVRES + '/'); duree = time.time() - debut_test
controle(essai.status_code == 200 and duree >= PAUSE * 0.9, 'Requête polie réussie, avec une pause de %.1f s' % duree, 'Attendu : code 200 et une pause d\'au moins PAUSE secondes')

✅ Requête polie réussie, avec une pause de 1.7 s


## Étape 8 — Votre script, version polie, sur 3 pages
On refait le crawling du notebook précédent, avec `get_poli` à la place de `requests.get`. Comparez la **durée** : c'est le prix de la politesse.

In [40]:
debut = time.time()
url, pages, nb_livres, nb_requetes = LIVRES + "/", 0, 0, 0
while url and pages < 3:
    reponse = get_poli(url)
    nb_requetes += 1
    page = BeautifulSoup(reponse.content, "html.parser")
    nb_livres += len(page.select("article.product_pod"))
    suivant = page.select_one("li.next a")
    url = urljoin(url, suivant["href"]) if suivant else None
    pages += 1
print(f"{nb_livres} livres, {nb_requetes} requêtes, {time.time() - debut:.1f} secondes")
print("Version classique : à vous de chronométrer la même chose avec requests.get, sans pause")

60 livres, 3 requêtes, 5.3 secondes
Version classique : à vous de chronométrer la même chose avec requests.get, sans pause


### Question
**Chronométrez la même collecte avec `requests.get` sans pause. Combien de fois plus lente est la version polie ? À ce rythme, combien de temps pour 1 000 pages ? Pourquoi est-ce acceptable, et que faire si vous devez vraiment aller plus vite ?**

*Votre réponse (2 à 4 lignes) :*

> 

In [41]:
import time

urls = [f"https://books.toscrape.com/catalogue/page-{i}.html" for i in range(1, 21)]

debut = time.time()
for u in urls:
    requests.get(u, timeout=20)
t_rapide = time.time() - debut

debut = time.time()
for u in urls:
    get_poli(u)
t_poli = time.time() - debut

print(f"sans pause : {t_rapide:.1f} s | polie : {t_poli:.1f} s | rapport : x{t_poli / t_rapide:.1f}")
print(f"estimation pour 1 000 pages : {t_rapide / 20 * 1000 / 60:.1f} min vs {t_poli / 20 * 1000 / 60:.1f} min")

sans pause : 15.0 s | polie : 36.4 s | rapport : x2.4
estimation pour 1 000 pages : 12.5 min vs 30.4 min


### Question
**Votre script pourrait-il ignorer robots.txt et le User-Agent ? Qu'est-ce que cela changerait pour le site, pour vous, et pour votre organisation ?**

*Votre réponse (2 à 4 lignes) :*

> 